# Modelica Dataset Explorer

This notebook demonstrates the **ModelicaDataset** API — a read-only Python wrapper
around the pipeline's SQLite databases.  It lets you:

- Browse sources, commits, and experiment classes.
- Walk the **semantic version timeline** of any model (only commits where the canonical form actually changed).
- Navigate forward/backward between distinct versions.
- Map between canonical snapshots and their originating commits.
- Read canonical `.mo` files and inspect validation metrics.
- Fetch **GitHub metadata** for commits: author info, PRs, linked issues, and full change context.

## 1. Setup

Import the API and open a connection.  The `ModelicaDataset` context manager
opens two read-only SQLite connections (main pipeline DB and Step 2 classes DB)
and closes them automatically when done.

In [ ]:
import sys
from pathlib import Path

# Ensure the dataset_creator package is importable
sys.path.insert(0, str(Path.cwd().parent))

from dataset.api import ModelicaDataset

ds = ModelicaDataset()
ds.open()

# List available sources
sources = ds.list_sources()
print(f"Available sources: {sources}")

SOURCE = sources[0] if sources else "MSL"
print(f"Using source: {SOURCE}")

## 2. Dataset Summary

A quick overview of what the dataset contains for the selected source:
total commits, canonical snapshots, unique snapshots, and experiment classes.

In [ ]:
stats = ds.summary(SOURCE)
for key, value in stats.items():
    print(f"  {key:25s}: {value:,}")

## 3. Listing Experiment Classes

These are the Modelica classes that have `experiment` annotations and
at least one canonical snapshot in the dataset.

In [ ]:
classes = ds.list_experiment_classes(SOURCE)
print(f"Total experiment classes: {len(classes)}\n")
print("First 10:")
for cls in classes[:10]:
    print(f"  {cls}")

# Pick one class for the following examples
EXAMPLE_CLASS = classes[0] if classes else "Modelica.Fluid.Examples.BranchingDynamicPipes"
print(f"\nUsing example class: {EXAMPLE_CLASS}")

## 4. Full Commit Timeline of a Class

The **class timeline** lists every commit that produced a canonical snapshot
for a given class, in repository order.  Note that many consecutive entries
may share the same `canonical_hash` — those are commits that touched the
model's dependencies but didn't change its semantic content.

In [ ]:
timeline = ds.get_class_timeline(SOURCE, EXAMPLE_CLASS)
print(f"Timeline length: {len(timeline)} entries\n")

print(f"{'#':>4}  {'Commit':10}  {'Canonical Hash':12}")
print("-" * 36)
for i, cv in enumerate(timeline[:15]):
    print(f"{i:4d}  {cv.commit_hash[:10]}  {cv.canonical_hash[:12]}")
if len(timeline) > 15:
    print(f"  ... and {len(timeline) - 15} more")

## 5. Semantic Versions (Distinct Canonical Changes)

The **semantic version** timeline collapses consecutive duplicate hashes.
Each entry is a *distinct* canonical form — a genuine semantic change.
This tells you how many times the model's actual behavior changed across
the repository's history.

In [ ]:
versions = ds.get_semantic_versions(SOURCE, EXAMPLE_CLASS)
print(f"Semantic versions: {len(versions)} distinct canonical forms\n")

print(f"{'V#':>3}  {'First Commit':10}  {'Hash':12}  {'Commits':>7}")
print("-" * 42)
for sv in versions[:20]:
    print(f"{sv.version_index:3d}  {sv.first_commit[:10]}  {sv.canonical_hash[:12]}  {sv.commit_count:7d}")
if len(versions) > 20:
    print(f"  ... and {len(versions) - 20} more")

## 6. Navigating Between Versions

Given any commit, you can jump to the **next** or **previous** semantic version.
This is useful for finding the exact commit that introduced a change.

In [ ]:
if versions:
    start = versions[0]
    print(f"Starting at version {start.version_index}: commit {start.first_commit[:10]}, hash {start.canonical_hash[:12]}")

    nxt = ds.get_next_version(SOURCE, EXAMPLE_CLASS, start.first_commit)
    if nxt:
        print(f"  → Next version {nxt.version_index}: commit {nxt.first_commit[:10]}, hash {nxt.canonical_hash[:12]}")

        prev = ds.get_previous_version(SOURCE, EXAMPLE_CLASS, nxt.first_commit)
        if prev:
            print(f"  ← Previous version {prev.version_index}: commit {prev.first_commit[:10]}, hash {prev.canonical_hash[:12]}")
    else:
        print("  (only one version exists)")
else:
    print("No versions available for this class.")

## 7. Mapping: Canonical Snapshot → Commits

Each deduplicated canonical snapshot can be traced back to **all originating commits**.
This shows which commits produced the exact same canonical model content.

In [ ]:
if versions:
    target_hash = versions[0].canonical_hash
    commits = ds.get_commits_for_canonical(SOURCE, target_hash)
    print(f"Canonical hash: {target_hash[:16]}...")
    print(f"Produced by {len(commits)} commit(s):")
    for c in commits[:10]:
        info = ds.get_commit(SOURCE, c)
        msg = (info.commit_message[:60] + '...') if info and len(info.commit_message) > 60 else (info.commit_message if info else '')
        print(f"  {c[:10]}  {msg}")
    if len(commits) > 10:
        print(f"  ... and {len(commits) - 10} more")

## 8. Mapping: Commit → Affected Models

Given a commit, find **all experiment models** that were affected (i.e. had
their canonical snapshot produced for that commit).

In [ ]:
# Pick a commit that touches our example class
touching = ds.get_commits_touching_class(SOURCE, EXAMPLE_CLASS)
if touching:
    sample_commit = touching[0]
    models = ds.get_models_for_commit(SOURCE, sample_commit)
    print(f"Commit {sample_commit[:10]} affects {len(models)} model(s):")
    for m in models[:15]:
        print(f"  {'[exp]' if m.is_experiment else '[sup]'} {m.class_name}")
    if len(models) > 15:
        print(f"  ... and {len(models) - 15} more")

## 9. Identifying Semantic-Change Commits

These are the commits where the model's canonical content **actually changed**,
as opposed to commits that touched surrounding code but left this model
semantically identical.

In [ ]:
change_commits = ds.get_semantic_change_commits(SOURCE, EXAMPLE_CLASS)
all_commits = ds.get_commits_touching_class(SOURCE, EXAMPLE_CLASS)

print(f"Commits touching this class   : {len(all_commits)}")
print(f"Commits with semantic changes  : {len(change_commits)}")
print(f"Commits with no semantic effect: {len(all_commits) - len(change_commits)}")

if change_commits:
    print(f"\nFirst 5 semantic-change commits:")
    for c in change_commits[:5]:
        info = ds.get_commit(SOURCE, c)
        msg = (info.commit_message[:50] + '...') if info and len(info.commit_message) > 50 else (info.commit_message if info else '')
        print(f"  {c[:10]}  {msg}")

## 10. Reading Canonical Model Content

Each canonical snapshot is a `.mo` file on disk.  The API can read it
directly so you can inspect or diff model versions.

In [ ]:
if versions:
    path = versions[0].canonical_model_path
    content = ds.read_canonical_model(path)
    if content:
        lines = content.splitlines()
        print(f"File: {path}")
        print(f"Total lines: {len(lines)}\n")
        print("First 20 lines:")
        print("-" * 60)
        for line in lines[:20]:
            print(line)
        if len(lines) > 20:
            print(f"... ({len(lines) - 20} more lines)")
    else:
        print(f"Canonical file not found on disk: {path}")

## 11. Validation Metrics

Each unique canonical snapshot has been validated with `checkModel()`.
The result includes compilation status, equation count, variable count,
and source-size metrics.

In [ ]:
if versions:
    result = ds.get_check_model_result(SOURCE, versions[0].canonical_hash)
    if result:
        print(f"Class       : {result.class_name}")
        print(f"Compiles    : {result.compiles}")
        print(f"Equations   : {result.equation_count:,}")
        print(f"Variables   : {result.variable_count:,}")
        print(f"LOC         : {result.line_count:,}")
        print(f"Bytes       : {result.byte_count:,}")
        if result.error_message:
            print(f"Error       : {result.error_message[:100]}")
    else:
        print("No checkModel result found for this snapshot.")

## 12. GitHub Commit Metadata

The API can fetch full commit details from GitHub — author, committer,
timestamp, diff stats, and the list of changed files. Set the
`GITHUB_TOKEN` environment variable to avoid rate limits.

> **Note:** These calls hit the live GitHub API. Each cell makes 1-3 requests.

In [ ]:
# Pick a semantic-change commit to inspect
change_commits = ds.get_semantic_change_commits(SOURCE, EXAMPLE_CLASS)
if change_commits:
    gh_commit_hash = change_commits[0]
    print(f"Fetching GitHub metadata for {gh_commit_hash[:10]}...\n")
    try:
        gh = ds.get_github_commit(SOURCE, gh_commit_hash)
        print(f"Author      : {gh.author_name} ({gh.author_login})")
        print(f"Date        : {gh.authored_date}")
        print(f"URL         : {gh.url}")
        print(f"Files changed: {gh.files_changed} (+{gh.additions} -{gh.deletions})")
        print(f"Message     : {gh.message[:120]}")
        mo_files = [f for f in gh.changed_files if f.endswith('.mo')]
        if mo_files:
            print(f"\n.mo files touched ({len(mo_files)}):")
            for f in mo_files[:10]:
                print(f"  {f}")
            if len(mo_files) > 10:
                print(f"  ... and {len(mo_files) - 10} more")
    except RuntimeError as e:
        print(f"GitHub API error: {e}")
else:
    print("No semantic change commits available.")

## 13. Associated Pull Requests

GitHub can tell us which pull request(s) a commit was part of,
including PR title, labels, and merge status.

In [ ]:
if change_commits:
    try:
        prs = ds.get_github_pull_requests(SOURCE, gh_commit_hash)
        if prs:
            for pr in prs:
                print(f"PR #{pr.number}: {pr.title}")
                print(f"  State: {pr.state}, Merged: {pr.merged}")
                print(f"  Labels: {', '.join(pr.labels) or '(none)'}")
                print(f"  URL: {pr.url}")
        else:
            print("No pull requests associated with this commit.")
    except RuntimeError as e:
        print(f"GitHub API error: {e}")

## 14. Linked Issues from Commit Message

The API parses `#NNN` references from the commit message and fetches
each referenced issue from GitHub. This connects semantic model changes
back to the bug reports or feature requests that motivated them.

In [ ]:
if change_commits:
    try:
        issues = ds.get_github_linked_issues(SOURCE, gh_commit_hash)
        if issues:
            for iss in issues:
                print(f"Issue #{iss.number}: {iss.title}")
                print(f"  State: {iss.state}, Labels: {', '.join(iss.labels) or '(none)'}")
                print(f"  URL: {iss.url}\n")
        else:
            print("No issue references found in the commit message.")
    except RuntimeError as e:
        print(f"GitHub API error: {e}")

## 15. Full Change Context for a Semantic Version

Combine everything: for a commit that changed a model's semantics,
fetch the GitHub commit, PRs, linked issues, and the local dataset
context (affected models, previous/next canonical version) in one call.

In [ ]:
if change_commits:
    try:
        ctx = ds.get_github_semantic_change_context(SOURCE, EXAMPLE_CLASS, gh_commit_hash)
        c = ctx['commit']
        print(f"=== Semantic Change Context ===")
        print(f"Commit       : {c.sha[:10]} by {c.author_name}")
        print(f"Date         : {c.authored_date}")
        print(f"Message      : {c.message[:100]}")
        print(f"Diff stats   : {ctx['total_files_changed']} files (+{ctx['additions']} -{ctx['deletions']})")
        print(f".mo files    : {len(ctx['mo_files_changed'])}")
        print(f"Models affected: {ctx['affected_models_count']}")
        print()
        if ctx['pull_requests']:
            for pr in ctx['pull_requests']:
                print(f"PR #{pr.number}: {pr.title} [{', '.join(pr.labels) or 'no labels'}]")
        if ctx['linked_issues']:
            for iss in ctx['linked_issues']:
                print(f"Issue #{iss.number}: {iss.title} ({iss.state})")
        print()
        if ctx['previous_version']:
            pv = ctx['previous_version']
            print(f"← Previous version (v{pv.version_index}): {pv.canonical_hash[:12]} from {pv.first_commit[:10]}")
        else:
            print("← (this is the first version)")
        if ctx['next_version']:
            nv = ctx['next_version']
            print(f"→ Next version (v{nv.version_index}): {nv.canonical_hash[:12]} from {nv.first_commit[:10]}")
        else:
            print("→ (this is the latest version)")
    except RuntimeError as e:
        print(f"GitHub API error: {e}")

## Cleanup

Close the database connections when done.

In [ ]:
ds.close()
print("Connections closed.")